# So sánh A/B: General mode vs IRAC mode

Eval trên **12 câu** (`test_subset_2mode.json`) phủ đủ 4 gap.  
Mỗi câu chạy **2 lần**: một lần `--response-mode general`, một lần `--response-mode irac`.  
Kết quả: F1 Khoản, F1 Điều, NormR per-question + aggregate.

> **Lưu ý**: cần Neo4j + Qdrant đang chạy và ANTHROPIC_API_KEY trong `.env`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root

from dotenv import load_dotenv
load_dotenv("../.env")

import json
import pandas as pd
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.3f}".format)

SUBSET_PATH = Path("../data/evaluation/test_subset_2mode.json")
OUT_DIR     = Path("../data/evaluation")
test_set    = json.loads(SUBSET_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(test_set)} câu từ {SUBSET_PATH.name}")
pd.DataFrame([{"id": q["id"], "gap": q["gap_type"], "question": q["question"][:70]} for q in test_set])

Loaded 12 câu từ test_subset_2mode.json


,id,gap,question
0,Q005,gap1,Những trường hợp đăng ký biến động đất đai nào phải cấp mới Giấy chứng
1,Q014,gap1,"Theo pháp luật đất đai hiện hành, ai được coi là cá nhân trực tiếp sản"
2,Q001,gap2,Hạn mức giao đất ở cho cá nhân tại TP.HCM tối đa là bao nhiêu m²?
3,Q002,gap2,Hạn mức giao đất ở cho cá nhân tại tỉnh Đồng Nai tối đa là bao nhiêu m
4,Q007,gap2,Mức phí thẩm định hồ sơ cấp Giấy chứng nhận quyền sử dụng đất tại TP.H
5,Q003,gap3,Cá nhân muốn chuyển mục đích sử dụng đất từ đất nông nghiệp sang đất ở
6,Q011,gap3,"Khi cá nhân được giao đất, cho thuê đất để sử dụng vào mục đích phi nô"
7,Q019,gap3,"Khi cá nhân được giao đất từ đất chuyên trồng lúa sang xây dựng nhà ở,"
8,Q022,gap4,Hồ sơ giao đất ở của tôi nộp tháng 6/2024 nhưng đến nay (tháng 10/2024
9,Q023,gap4,Hồ sơ cấp Giấy chứng nhận quyền sử dụng đất nộp năm 2023 nhưng chưa gi


In [2]:
# ---------------------------------------------------------------------------
# Khởi tạo shared clients (load 1 lần, dùng cho cả 2 mode)
# ---------------------------------------------------------------------------
from src.evaluation.run_evaluation import _build_shared_clients
print("Đang load Neo4j + Qdrant + BGE-M3 (30-60s lần đầu)...")
clients = _build_shared_clients()
print("✅ Clients ready")

Đang load Neo4j + Qdrant + BGE-M3 (30-60s lần đầu)...


INFO HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
INFO TensorFlow version 2.19.0 available.
INFO Loading model BAAI/bge-m3 ...
INFO No device provided, using mps
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO Loading SentenceTransformer model from BAAI/bge-m3.
INFO HTTP Request: HEAD https:

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/main "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors.index.j

✅ Clients ready


In [3]:
# ---------------------------------------------------------------------------
# Hàm helper: chạy 1 mode, trả về DataFrame per-question
# ---------------------------------------------------------------------------
from src.evaluation.run_evaluation import run_system_on_test_set

LLM_CACHE = OUT_DIR / ".llm_cache"

def run_mode(mode: str) -> pd.DataFrame:
    """Chạy graphrag với mode chỉ định, trả về DataFrame per-question."""
    print(f"\n{'='*60}")
    print(f"  Chạy mode: {mode.upper()}  ({len(test_set)} câu)")
    print(f"{'='*60}")
    results = run_system_on_test_set(
        test_set,
        system="graphrag",
        clients=clients,
        llm_cache_dir=LLM_CACHE,
        response_mode=mode,
    )
    rows = []
    for r in results:
        rows.append({
            "id":       r["id"],
            "gap":      r["gap_type"],
            "question": r["question"][:55],
            "F1_kh":    round(r["citation_score"]["f1"], 3),
            "F1_di":    round(r["citation_score_dieu"]["f1"], 3),
            "NormR":    round(r["norm_recall"], 3),
            "#pred":    len(r["pred_citations"]),
            "#gt":      len(r["ground_truth_citations"]),
            "neg_ok":   r["negative_correct"],
            "elapsed":  r["elapsed_seconds"],
            "mode":     r.get("response_mode", mode),
        })
    df = pd.DataFrame(rows)
    # Lưu raw results JSON để dùng compare_runs nếu cần
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    out = OUT_DIR / f"results_graphrag_{mode}_{ts}.json"
    out.write_text(json.dumps(
        {"system": "graphrag", "mode": mode, "test_set": str(SUBSET_PATH),
         "timestamp": ts, "results": results},
        ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  → Saved: {out.name}")
    return df

In [4]:
# ---------------------------------------------------------------------------
# Chạy GENERAL mode
# ---------------------------------------------------------------------------
df_general = run_mode("general")
df_general

INFO [graphrag] 1/12 Q005: Những trường hợp đăng ký biến động đất đai nào phải cấp mới ...
INFO run_pipeline: plan_query cho 'Những trường hợp đăng ký biến động đất đai nào phải cấp mới ...'



  Chạy mode: GENERAL  (12 câu)


INFO HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO plan_query | theme=dat-dai procedure=None jurisdiction=None temporal=None is_complete=False missing=['procedure', 'jurisdiction'] temporal_ctx=False
INFO run_pipeline: plan=dat-dai/None complete=False
INFO run_pipeline: response_mode='general'
INFO run_pipeline: force_jurisdiction='toan-quoc' áp dụng, is_complete=False
INFO run_pipeline: bypass_completeness=True — bỏ qua missing=['procedure'], tiếp tục retrieval với best-effort
INFO run_pipeline: extract_subgraph


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"
INFO Stage 1: top-5 scores=[0.674, 0.637, 0.598, 0.596, 0.595], threshold=0.3 → 5 norm_ids = ['nghi-dinh-101-2024-nd-cp', 'nghi-dinh-226-2025-nd-cp', 'nghi-dinh-151-2025-nd-cp', 'nghi-dinh-102-2024-nd-cp', 'nghi-dinh-49-2026-nd-cp']
INFO Stage 2 (norm_ids): 9 norms (jurisdiction=toan-quoc, temporal=None): ['nghi-dinh-112-2024-nd-cp', 'luat-dat-dai-2024', 'nghi-dinh-151-2025-nd-cp', 'nghi-quyet-254-2025-qh15', 'nghi-dinh-226-2025-nd-cp', 'nghi-dinh-101-2024-nd-cp', 'nghi-dinh-49-2026-nd-cp', 'nghi-dinh-102-2024-nd-cp', 'nghi-dinh-50-2026-nd-cp']
INFO run_pipeline: 9 norm_ids, 0 graph_comp_ids từ Stage 2+3
INFO run_pipeline: hybrid_search


...

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"
INFO Stage 1: top-5 scores=[0.574, 0.568, 0.565, 0.557, 0.515], threshold=0.3 → 5 norm_ids = ['nghi-quyet-02-2023-nq-hdnd-tp-hcm', 'nghi-quyet-22-2024-nq-hdnd-dong-nai', 'nghi-quyet-21-2024-nq-hdnd-dong-nai', 'quyet-dinh-52-2016-qd-ubnd-tp-hcm', 'nghi-dinh-226-2025-nd-cp']
INFO Stage 2 (norm_ids): 9 norms (jurisdiction=toan-quoc, temporal=None): ['nghi-dinh-112-2024-nd-cp', 'luat-dat-dai-2024', 'nghi-dinh-151-2025-nd-cp', 'nghi-quyet-254-2025-qh15', 'nghi-dinh-226-2025-nd-cp', 'nghi-dinh-101-2024-nd-cp', 'nghi-dinh-49-2026-nd-cp', 'nghi-dinh-102-2024-nd-cp', 'nghi-dinh-50-2026-nd-cp']
INFO run_pipeline: 9 norm_ids, 0 graph_comp_ids từ Stage 2+3
INFO run_pipeline: hybrid_search


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"
INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/scroll "HTTP/1.1 200 OK"
INFO hybrid_search: dense=50, keyword=0, graph=0 candidates
INFO hybrid_search: top-14 | pass-1(struct-cite)=0, pass-0.5(label-keyword)=0, pass0(dense-floor)=7, pass1(rrf-breadth)=0, pass2(depth)=7 | caps: per_norm=3, per_tier={1: 8, 2: 8, 3: 6, 4: 8} | best rrf=0.0400 | tier_dist={1: 6, 2: 8} | norm_dist={'nghi-dinh-101-2024-nd-cp': 3, 'luat-dat-dai-2024': 3, 'nghi-quyet-254-2025-qh15': 3, 'nghi-dinh-102-2024-nd-cp': 1, 'nghi-dinh-50-2026-nd-cp': 2, 'nghi-dinh-151-2025-nd-cp': 1, 'nghi-dinh-49-2026-nd-cp': 1}
INFO run_pipeline: 14 scored units
INFO run_pipeline: assemble_context
INFO assemble_context: 14 blocks, ~4514 tokens
INFO run_pipeline: generate_answer
INFO HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO generate_answer: 179 chars, 0 citations, se

  → Saved: results_graphrag_general_20260530-115843.json


,id,gap,question,F1_kh,F1_di,NormR,#pred,#gt,neg_ok,elapsed,mode
0,Q005,gap1,Những trường hợp đăng ký biến động đất đai nào phải cấp,0.400,0.400,1.000,4,1,None,15.770,general
1,Q014,gap1,"Theo pháp luật đất đai hiện hành, ai được coi là cá nhâ",0.400,0.400,1.000,4,1,None,14.020,general
2,Q001,gap2,Hạn mức giao đất ở cho cá nhân tại TP.HCM tối đa là bao,1.000,1.000,1.000,3,3,None,13.010,general
3,Q002,gap2,Hạn mức giao đất ở cho cá nhân tại tỉnh Đồng Nai tối đa,0.857,0.857,1.000,3,4,None,14.820,general
4,Q007,gap2,Mức phí thẩm định hồ sơ cấp Giấy chứng nhận quyền sử dụ,0.500,0.500,1.000,2,2,None,16.510,general
5,Q003,gap3,Cá nhân muốn chuyển mục đích sử dụng đất từ đất nông ng,0.000,0.286,0.667,4,3,None,24.990,general
6,Q011,gap3,"Khi cá nhân được giao đất, cho thuê đất để sử dụng vào",0.400,0.400,0.333,2,3,None,19.660,general
7,Q019,gap3,Khi cá nhân được giao đất từ đất chuyên trồng lúa sang,0.571,0.571,0.500,2,5,None,21.190,general
8,Q022,gap4,Hồ sơ giao đất ở của tôi nộp tháng 6/2024 nhưng đến nay,0.000,0.000,0.500,1,2,None,17.710,general
9,Q023,gap4,Hồ sơ cấp Giấy chứng nhận quyền sử dụng đất nộp năm 202,0.333,0.333,1.000,3,3,None,23.040,general


In [5]:
# ---------------------------------------------------------------------------
# Chạy IRAC mode
# ---------------------------------------------------------------------------
df_irac = run_mode("irac")
df_irac

INFO [graphrag] 1/12 Q005: Những trường hợp đăng ký biến động đất đai nào phải cấp mới ...
INFO run_pipeline: plan_query cho 'Những trường hợp đăng ký biến động đất đai nào phải cấp mới ...'
INFO plan_query: cache HIT (48569ce4a11d380d) — $0 API
INFO plan_query | theme=dat-dai procedure=None jurisdiction=None temporal=None is_complete=False missing=['procedure', 'jurisdiction'] temporal_ctx=False
INFO run_pipeline: plan=dat-dai/None complete=False
INFO run_pipeline: response_mode='irac'
INFO run_pipeline: force_jurisdiction='toan-quoc' áp dụng, is_complete=False
INFO run_pipeline: bypass_completeness=True — bỏ qua missing=['procedure'], tiếp tục retrieval với best-effort
INFO run_pipeline: extract_subgraph



  Chạy mode: IRAC  (12 câu)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"
INFO Stage 1: top-5 scores=[0.674, 0.637, 0.598, 0.596, 0.595], threshold=0.3 → 5 norm_ids = ['nghi-dinh-101-2024-nd-cp', 'nghi-dinh-226-2025-nd-cp', 'nghi-dinh-151-2025-nd-cp', 'nghi-dinh-102-2024-nd-cp', 'nghi-dinh-49-2026-nd-cp']
INFO Stage 2 (norm_ids): 9 norms (jurisdiction=toan-quoc, temporal=None): ['nghi-dinh-112-2024-nd-cp', 'luat-dat-dai-2024', 'nghi-dinh-151-2025-nd-cp', 'nghi-quyet-254-2025-qh15', 'nghi-dinh-226-2025-nd-cp', 'nghi-dinh-101-2024-nd-cp', 'nghi-dinh-49-2026-nd-cp', 'nghi-dinh-102-2024-nd-cp', 'nghi-dinh-50-2026-nd-cp']
INFO run_pipeline: 9 norm_ids, 0 graph_comp_ids từ Stage 2+3
INFO run_pipeline: hybrid_search


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"
INFO Stage 1: top-5 scores=[0.574, 0.568, 0.565, 0.557, 0.515], threshold=0.3 → 5 norm_ids = ['nghi-quyet-02-2023-nq-hdnd-tp-hcm', 'nghi-quyet-22-2024-nq-hdnd-dong-nai', 'nghi-quyet-21-2024-nq-hdnd-dong-nai', 'quyet-dinh-52-2016-qd-ubnd-tp-hcm', 'nghi-dinh-226-2025-nd-cp']
INFO Stage 2 (norm_ids): 9 norms (jurisdiction=toan-quoc, temporal=None): ['nghi-dinh-112-2024-nd-cp', 'luat-dat-dai-2024', 'nghi-dinh-151-2025-nd-cp', 'nghi-quyet-254-2025-qh15', 'nghi-dinh-226-2025-nd-cp', 'nghi-dinh-101-2024-nd-cp', 'nghi-dinh-49-2026-nd-cp', 'nghi-dinh-102-2024-nd-cp', 'nghi-dinh-50-2026-nd-cp']
INFO run_pipeline: 9 norm_ids, 0 graph_comp_ids từ Stage 2+3
INFO run_pipeline: hybrid_search


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"
INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/scroll "HTTP/1.1 200 OK"
INFO hybrid_search: dense=50, keyword=0, graph=0 candidates
INFO hybrid_search: top-14 | pass-1(struct-cite)=0, pass-0.5(label-keyword)=0, pass0(dense-floor)=7, pass1(rrf-breadth)=0, pass2(depth)=7 | caps: per_norm=3, per_tier={1: 8, 2: 8, 3: 6, 4: 8} | best rrf=0.0400 | tier_dist={1: 6, 2: 8} | norm_dist={'nghi-dinh-101-2024-nd-cp': 3, 'luat-dat-dai-2024': 3, 'nghi-quyet-254-2025-qh15': 3, 'nghi-dinh-102-2024-nd-cp': 1, 'nghi-dinh-50-2026-nd-cp': 2, 'nghi-dinh-151-2025-nd-cp': 1, 'nghi-dinh-49-2026-nd-cp': 1}
INFO run_pipeline: 14 scored units
INFO run_pipeline: assemble_context
INFO assemble_context: 14 blocks, ~4514 tokens
INFO run_pipeline: generate_answer
INFO HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO generate_answer: 179 chars, 0 citations, se

  → Saved: results_graphrag_irac_20260530-120234.json


,id,gap,question,F1_kh,F1_di,NormR,#pred,#gt,neg_ok,elapsed,mode
0,Q005,gap1,Những trường hợp đăng ký biến động đất đai nào phải cấp,0.400,0.400,1.000,4,1,None,15.490,irac
1,Q014,gap1,"Theo pháp luật đất đai hiện hành, ai được coi là cá nhâ",0.400,0.400,1.000,4,1,None,10.120,irac
2,Q001,gap2,Hạn mức giao đất ở cho cá nhân tại TP.HCM tối đa là bao,1.000,1.000,1.000,3,3,None,10.760,irac
3,Q002,gap2,Hạn mức giao đất ở cho cá nhân tại tỉnh Đồng Nai tối đa,0.857,0.857,1.000,3,4,None,16.490,irac
4,Q007,gap2,Mức phí thẩm định hồ sơ cấp Giấy chứng nhận quyền sử dụ,0.000,0.000,0.000,0,2,None,16.130,irac
5,Q003,gap3,Cá nhân muốn chuyển mục đích sử dụng đất từ đất nông ng,0.000,0.222,0.667,6,3,None,34.180,irac
6,Q011,gap3,"Khi cá nhân được giao đất, cho thuê đất để sử dụng vào",0.857,0.857,1.000,4,3,None,30.090,irac
7,Q019,gap3,Khi cá nhân được giao đất từ đất chuyên trồng lúa sang,0.250,0.250,0.250,3,5,None,26.900,irac
8,Q022,gap4,Hồ sơ giao đất ở của tôi nộp tháng 6/2024 nhưng đến nay,0.000,0.400,1.000,3,2,None,20.090,irac
9,Q023,gap4,Hồ sơ cấp Giấy chứng nhận quyền sử dụng đất nộp năm 202,0.400,0.400,1.000,2,3,None,24.600,irac


In [7]:
# ---------------------------------------------------------------------------
# So sánh per-question: Δ F1_kh = irac - general
# ---------------------------------------------------------------------------
df_cmp = df_general[["id","gap","question","F1_kh","F1_di","NormR","#pred"]].copy()
df_cmp.columns = ["id","gap","question","G_F1kh","G_F1di","G_NormR","G_pred"]

df_cmp["I_F1kh"]  = df_irac["F1_kh"].values
df_cmp["I_F1di"]  = df_irac["F1_di"].values
df_cmp["I_NormR"] = df_irac["NormR"].values
df_cmp["I_pred"]  = df_irac["#pred"].values
df_cmp["ΔF1kh"]   = (df_cmp["I_F1kh"] - df_cmp["G_F1kh"]).round(3)

def delta_label(v):
    if v > 0.05:  return f"▲ {v:+.3f}"
    if v < -0.05: return f"▼ {v:+.3f}"
    return f"  {v:+.3f}"

df_cmp["Δ_label"] = df_cmp["ΔF1kh"].apply(delta_label)

print("Per-question: G = General | I = IRAC | Δ = I − G  (▲ irac tốt hơn / ▼ irac tệ hơn)")
df_cmp[["id","gap","question","G_F1kh","I_F1kh","Δ_label","G_NormR","I_NormR","G_pred","I_pred"]]

Per-question: G = General | I = IRAC | Δ = I − G  (▲ irac tốt hơn / ▼ irac tệ hơn)


,id,gap,question,G_F1kh,I_F1kh,Δ_label,G_NormR,I_NormR,G_pred,I_pred
0,Q005,gap1,Những trường hợp đăng ký biến động đất đai nào phải cấp,0.400,0.400,+0.000,1.000,1.000,4,4
1,Q014,gap1,"Theo pháp luật đất đai hiện hành, ai được coi là cá nhâ",0.400,0.400,+0.000,1.000,1.000,4,4
2,Q001,gap2,Hạn mức giao đất ở cho cá nhân tại TP.HCM tối đa là bao,1.000,1.000,+0.000,1.000,1.000,3,3
3,Q002,gap2,Hạn mức giao đất ở cho cá nhân tại tỉnh Đồng Nai tối đa,0.857,0.857,+0.000,1.000,1.000,3,3
4,Q007,gap2,Mức phí thẩm định hồ sơ cấp Giấy chứng nhận quyền sử dụ,0.500,0.000,▼ -0.500,1.000,0.000,2,0
5,Q003,gap3,Cá nhân muốn chuyển mục đích sử dụng đất từ đất nông ng,0.000,0.000,+0.000,0.667,0.667,4,6
6,Q011,gap3,"Khi cá nhân được giao đất, cho thuê đất để sử dụng vào",0.400,0.857,▲ +0.457,0.333,1.000,2,4
7,Q019,gap3,Khi cá nhân được giao đất từ đất chuyên trồng lúa sang,0.571,0.250,▼ -0.321,0.500,0.250,2,3
8,Q022,gap4,Hồ sơ giao đất ở của tôi nộp tháng 6/2024 nhưng đến nay,0.000,0.000,+0.000,0.500,1.000,1,3
9,Q023,gap4,Hồ sơ cấp Giấy chứng nhận quyền sử dụng đất nộp năm 202,0.333,0.400,▲ +0.067,1.000,1.000,3,2


In [8]:
# ---------------------------------------------------------------------------
# Aggregate: mean F1 theo mode × gap
# ---------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

def agg(df, mode_label):
    rows = []
    # Overall (loại negative)
    sub = df[df["gap"] != "negative"]
    rows.append({"mode": mode_label, "gap": "OVERALL (non-neg)",
                 "F1_kh": sub["F1_kh"].mean(), "F1_di": sub["F1_di"].mean(),
                 "NormR": sub["NormR"].mean(), "n": len(sub)})
    # Per gap
    for gap in ["gap1","gap2","gap3","gap4"]:
        sub = df[df["gap"] == gap]
        if len(sub):
            rows.append({"mode": mode_label, "gap": gap,
                         "F1_kh": sub["F1_kh"].mean(), "F1_di": sub["F1_di"].mean(),
                         "NormR": sub["NormR"].mean(), "n": len(sub)})
    # Negative
    sub = df[df["gap"] == "negative"]
    rows.append({"mode": mode_label, "gap": "negative",
                 "F1_kh": None, "F1_di": None, "NormR": None,
                 "neg_ok": df_general[df_general["gap"]=="negative"]["neg_ok"].tolist(), "n": len(sub)})
    return pd.DataFrame(rows)

agg_df = pd.concat([agg(df_general, "general"), agg(df_irac, "irac")])
print("Aggregate F1 theo mode × gap:")
agg_df.set_index(["mode","gap"])[["F1_kh","F1_di","NormR","n"]]

Aggregate F1 theo mode × gap:


F1_kh  F1_di  NormR   n
mode    gap                                       
general OVERALL (non-neg)  0.466  0.492  0.818  11
        gap1               0.400  0.400  1.000   2
        gap2               0.786  0.786  1.000   3
        gap3               0.324  0.419  0.500   3
        gap4               0.333  0.333  0.833   3
        negative             NaN    NaN    NaN   1
irac    OVERALL (non-neg)  0.439  0.496  0.811  11
        gap1               0.400  0.400  1.000   2
        gap2               0.619  0.619  0.667   3
        gap3               0.369  0.443  0.639   3
        gap4               0.356  0.489  1.000   3
        negative             NaN    NaN    NaN   1

In [9]:
# ---------------------------------------------------------------------------
# Tóm tắt cuối: win/loss/tie giữa 2 mode
# ---------------------------------------------------------------------------
delta = df_cmp["ΔF1kh"]
wins   = (delta > 0.05).sum()
losses = (delta < -0.05).sum()
ties   = len(delta) - wins - losses

g_mean = df_general[df_general["gap"]!="negative"]["F1_kh"].mean()
i_mean = df_irac[df_irac["gap"]!="negative"]["F1_kh"].mean()

print(f"""\n{'='*55}
 TỔNG KẾT A/B  (12 câu / 4 gap)
{'='*55}
 General F1 Khoản (mean): {g_mean:.3f}
 IRAC    F1 Khoản (mean): {i_mean:.3f}
 Δ (IRAC − General):      {i_mean - g_mean:+.3f}

 Per-question  (|Δ| > 0.05):
   IRAC wins : {wins}
   IRAC losses: {losses}
   Ties       : {ties}
{'='*55}""")

# Top differences
print("\nCâu thay đổi nhiều nhất (|Δ| > 0.1):")
df_cmp[abs(df_cmp["ΔF1kh"]) > 0.1][["id","gap","question","G_F1kh","I_F1kh","ΔF1kh"]]


 TỔNG KẾT A/B  (12 câu / 4 gap)
 General F1 Khoản (mean): 0.466
 IRAC    F1 Khoản (mean): 0.439
 Δ (IRAC − General):      -0.027

 Per-question  (|Δ| > 0.05):
   IRAC wins : 2
   IRAC losses: 2
   Ties       : 8

Câu thay đổi nhiều nhất (|Δ| > 0.1):


,id,gap,question,G_F1kh,I_F1kh,ΔF1kh
4,Q007,gap2,Mức phí thẩm định hồ sơ cấp Giấy chứng nhận quyền sử dụ,0.500,0.000,-0.500
6,Q011,gap3,"Khi cá nhân được giao đất, cho thuê đất để sử dụng vào",0.400,0.857,0.457
7,Q019,gap3,Khi cá nhân được giao đất từ đất chuyên trồng lúa sang,0.571,0.250,-0.321
